# RDFLib graphs with pyling

This notebook shows the RDFLib integration path: build an RDFLib graph, pass it to pyling, and inspect the inferred closure as another RDFLib graph.

In [1]:
from rdflib import Graph, Namespace, RDF
from pyling import reason_stream

EX = Namespace("http://example.org/")

graph = Graph()
graph.bind("ex", EX)
graph.add((EX.Socrates, RDF.type, EX.Man))
graph.add((EX.Man, EX.subClassOf, EX.Mortal))

rules = """
@prefix : <http://example.org/> .
{ ?x a ?class . ?class :subClassOf ?superClass . } => { ?x a ?superClass } .
"""

result = reason_stream(
    {"sources": [graph.serialize(format="turtle"), rules]},
    rdf=True,
    include_input_facts_in_closure=True,
)
closure = result.as_rdflib_graph(include_input_facts=True)

print(len(graph), "input triples")
print(len(result.derived), "derived triples")
print((EX.Socrates, RDF.type, EX.Mortal) in closure)

2 input triples
1 derived triples
True


In [2]:
print(result.closure_n3)

@prefix : <http://example.org/> .
@prefix ex: <http://example.org/> .

:Man :subClassOf :Mortal .
:Socrates a :Man .
:Socrates a :Mortal .

